<a href="https://colab.research.google.com/github/althafceo/ITA0616-ML/blob/main/id3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import csv
import math
from collections import Counter


def load_data(filename):
    """Read CSV into a list of dicts; the last column is the class label."""
    with open(filename, newline="") as f:
        rows = list(csv.DictReader(f))
    target = list(rows[0].keys())[-1]
    return rows, target


def entropy(rows, target):
    counts = Counter(r[target] for r in rows)
    total = len(rows)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def info_gain(rows, attr, target):
    total = len(rows)
    remainder = 0.0
    for value in {r[attr] for r in rows}:
        subset = [r for r in rows if r[attr] == value]
        remainder += (len(subset) / total) * entropy(subset, target)
    return entropy(rows, target) - remainder


def id3(rows, attrs, target):
    labels = [r[target] for r in rows]

    # Case 1: all examples have the same class -> leaf
    if len(set(labels)) == 1:
        return labels[0]
    # Case 2: no attributes left -> majority class leaf
    if not attrs:
        return Counter(labels).most_common(1)[0][0]

    # Pick the attribute with the highest information gain
    gains = {a: info_gain(rows, a, target) for a in attrs}
    best = max(gains, key=gains.get)

    tree = {best: {}}
    remaining = [a for a in attrs if a != best]
    for value in {r[best] for r in rows}:
        subset = [r for r in rows if r[best] == value]
        tree[best][value] = id3(subset, remaining, target)
    return tree


def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(indent + "-> " + str(tree))
        return
    attr = next(iter(tree))
    for value, subtree in tree[attr].items():
        print(f"{indent}[{attr} = {value}]")
        print_tree(subtree, indent + "    ")


def classify(tree, sample):
    while isinstance(tree, dict):
        attr = next(iter(tree))
        tree = tree[attr][sample[attr]]
    return tree


if __name__ == "__main__":
    data, target = load_data("id3.csv")
    attributes = [a for a in data[0].keys() if a != target]

    print(f"Entropy of the whole dataset: {entropy(data, target):.4f}\n")
    print("Information gain at the root:")
    for a in attributes:
        print(f"  {a:<12}: {info_gain(data, a, target):.4f}")

    tree = id3(data, attributes, target)
    print("\nDecision tree (dictionary form):")
    print(tree)
    print("\nDecision tree (readable form):")
    print_tree(tree)

    new_sample = {"Outlook": "Sunny", "Temperature": "Cool",
                  "Humidity": "High", "Wind": "Strong"}
    print("\nNew sample:", new_sample)
    print("Predicted PlayTennis:", classify(tree, new_sample))

Entropy of the whole dataset: 0.9403

Information gain at the root:
  Outlook     : 0.2467
  Temperature : 0.0292
  Humidity    : 0.1518
  Wind        : 0.0481

Decision tree (dictionary form):
{'Outlook': {'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}, 'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes'}}

Decision tree (readable form):
[Outlook = Rain]
    [Wind = Weak]
        -> Yes
    [Wind = Strong]
        -> No
[Outlook = Sunny]
    [Humidity = High]
        -> No
    [Humidity = Normal]
        -> Yes
[Outlook = Overcast]
    -> Yes

New sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted PlayTennis: No
